In [ ]:
!pip install -qq pytabkit

In [ ]:
# %load_ext cudf.pandas

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import pandas as pd, numpy as np

train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
print(f'Train Shape: {train.shape}')
display(train.head(3))
print(f'\nTest Shape: {test.shape}')
display(test.head(3))

In [ ]:
train.nunique()

In [ ]:
TARGET = 'Heart Disease'
BASE = [col for col in train.columns if col not in ['id', TARGET]]
print(len(BASE), 'Base Features.')

In [ ]:
train[TARGET] = train[TARGET].map({'Absence': 0, 'Presence': 1})

print('Mapping applied: Absence=0, Presence=1')
display(train[[TARGET]].head(3))

# Feature Engineering

In [ ]:
combined_df = pd.concat([train, test], axis=0, ignore_index=True)

# DIGIT

In [ ]:
DIGIT = []

for col in BASE:
    max_val = combined_df[col].abs().max()
    if max_val == 0:
        max_int_digits = 1
    else:
        max_int_digits = int(np.log10(max_val)) + 1
    
    for i in range(max_int_digits):
        new_col = f"{col}_digit_pos_{i+1}"
        
        combined_df[new_col] = (combined_df[col] // (10**i)) % 10
        combined_df[new_col] = combined_df[new_col].astype(int)
        
        DIGIT.append(new_col)
    
    if combined_df[col].dtype == 'float64':
        max_dec_digits = 2
        
        if (combined_df[col] % 1 != 0).any():
            for i in range(1, max_dec_digits + 1):
                new_col = f"{col}_digit_dec_{i}"
                
                combined_df[new_col] = (combined_df[col] * (10**i)).round().astype(int) % 10
                
                DIGIT.append(new_col)

print(f"{len(DIGIT)} Digit Features Created!!")
print(f"Sample features: {DIGIT[:5]} ...")

# BIN Feature

In [ ]:
BINNING = []

target_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']

print(f"Target Features for Binning: {target_cols}")

for col in target_cols:
    qs = [4, 5, 10, 20]
    
    for q in qs:
        new_col = f"{col}_bin_q{q}"
        try:
            combined_df[new_col] = pd.qcut(combined_df[col], q=q, labels=False, duplicates='drop')
            BINNING.append(new_col)
        except ValueError:
            print(f"Skipped qcut {q} for {col} (Unique values too low)")

    # --- 2. Uniform Binning ---
    bins_list = [5, 10]
    
    for b in bins_list:
        new_col = f"{col}_bin_cut{b}"
        combined_df[new_col] = pd.cut(combined_df[col], bins=b, labels=False)
        BINNING.append(new_col)

    # --- 3. Custom Rounding Bins ---    
    if col == 'Age':
        new_col = f"{col}_bin_round5"
        combined_df[new_col] = (combined_df[col] / 5).round().astype(int)
        BINNING.append(new_col)
        
    if col in ['BP', 'Max HR']:
        new_col = f"{col}_bin_round10"
        combined_df[new_col] = (combined_df[col] / 10).round().astype(int)
        BINNING.append(new_col)

    if col == 'Cholesterol':
        new_col = f"{col}_bin_round50"
        combined_df[new_col] = (combined_df[col] / 50).round().astype(int)
        BINNING.append(new_col)

for col in BINNING:
    combined_df[col] = combined_df[col].fillna(-1).astype(int)

print(f"{len(BINNING)} Binning Features Created!!")
print(f"Sample: {BINNING[:5]} ...")

# ALL CATS

In [ ]:
ALL_CATS = []

for c in BASE:
    nc = f'{c}_cat'
    combined_df[nc] = combined_df[c].astype(str)
    ALL_CATS.append(nc)

# ALL FEATURE

In [ ]:
FEATURES = (
    BASE
    # DIGIT 
    # + BINNING 
    + ALL_CATS
)

In [ ]:
# ------------------------------------------------------------
# 0) Preconditions (Data Setup)
# ------------------------------------------------------------
if 'cudf' in str(type(combined_df)):
    combined_df = combined_df.to_pandas()

str_cols = [c for c in combined_df.columns if combined_df[c].dtype == "object"]
CAT_COLS = str_cols.copy()

for col in CAT_COLS:
    combined_df[col] = combined_df[col].astype(str).astype("category")

# Split back
train = combined_df.iloc[: len(train)].reset_index(drop=True)
test  = combined_df.iloc[len(train) :].reset_index(drop=True)

# Model

In [ ]:
import torch
import gc
from pytabkit import RealMLP_TD_Regressor 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from cuml.preprocessing import TargetEncoder

teacher_oof_df = pd.read_csv('/kaggle/input/datasets/masayakawamata/s6e2-realmlpdistil/oof_realmlp_distil.csv')
print(f"train length: {len(train)}")
print(f"teacher OOF length: {len(teacher_oof_df)}")

teacher_oof_df = teacher_oof_df.rename(columns={TARGET: 'teacher_pred'})
train = train.merge(teacher_oof_df[['id', 'teacher_pred']], on='id', how='left')
train['teacher_pred'] = train['teacher_pred'].fillna(train[TARGET])
teacher_preds = train['teacher_pred'].values

common_fixed_params = {
    "act": "mish",
    "ls_eps_sched": "coslog4",
    "p_drop_sched": "flat_cos",
    "wd_sched": "flat_cos",
    "bias_wd_factor": 0.0,
    "weight_param": "ntk",
    "bias_lr_factor": 0.1,
    "use_parametric_act": True,
    "add_front_scale": True,
    "act_lr_factor": 0.1,
    "block_str": "w-b-a-d",
    "hidden_sizes": "rectangular",
    "bias_init_mode": "he+5",
    "weight_init_mode": "std",
    "val_metric_name": "rmse", 
    "num_emb_type": "pbld",
    "batch_size": 1024,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "random_state": 42,
    "n_epochs": 32,
    "early_stopping_additive_patience": 40,
    "early_stopping_multiplicative_patience": 3,
    "n_ens": 8,
    "ens_av_before_softmax": False,
    "verbosity": 2,
}

top3_params = [
    {
    **common_fixed_params, 'n_hidden_layers': 3, 'hidden_width': 768, 'embedding_size': 4, 'lr': 0.02868285270525024, 'wd': 0.005511866840587377, 'p_drop': 0.09235935010477037, 'scale_lr_factor': 9.476033947354969, 'use_ls': True, 'ls_eps': 0.00572740420529087,
    'plr_sigma': 2.240745312676253,'plr_lr_factor': 0.06877512174484275,'plr_hidden_1': 4,'plr_hidden_2': 16,'sq_mom': 0.9296248741430283,'first_layer_lr_factor': 1.6707119602718028,
    'use_early_stopping': False,'max_one_hot_cat_size': 10
    },
    {
    **common_fixed_params,
    'n_hidden_layers': 3,'hidden_width': 768,'embedding_size': 4,'lr': 0.019183520450074383,'wd': 0.009859393010616932,'p_drop': 0.026105580772607273,'scale_lr_factor': 7.853056466117382,
    'use_ls': True,'ls_eps': 0.047925848795263626,'plr_sigma': 3.310419278751393,'plr_lr_factor': 0.07652536370654753,'plr_hidden_1': 4,'plr_hidden_2': 16,
    'sq_mom': 0.9138661522848395,'first_layer_lr_factor': 1.3848922234454992,'use_early_stopping': False,'max_one_hot_cat_size': 8
    },
    {
    **common_fixed_params,
    'n_hidden_layers': 2,'hidden_width': 768,'embedding_size': 16,'lr': 0.04271624411220208,'wd': 0.010890846606723997,'p_drop': 0.009467150649812572,'scale_lr_factor': 9.889496429982614,
    'use_ls': True,'ls_eps': 0.01931685296061534,'plr_sigma': 1.139116674110427,'plr_lr_factor': 0.02109852472356917,'plr_hidden_1': 16,'plr_hidden_2': 8,
    'sq_mom': 0.9368364937657381,'first_layer_lr_factor': 1.9302373368146657,'use_early_stopping': False,'max_one_hot_cat_size': 11
    }
]

oof_preds = np.zeros(len(train), dtype=np.float32)
test_preds = np.zeros(len(test), dtype=np.float32)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ALPHA = 0.5 

print(f"\nStarting Ensemble Training with Knowledge Distillation (Top 3 Models x 5 Folds)...")
print(f"{'Fold':<5} | {'Model':<5} | {'AUC':<10}")
print("-" * 35)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train[TARGET])):

    # Data Split
    X_tr = train.loc[tr_idx, FEATURES].copy()
    y_tr_hard = train.loc[tr_idx, TARGET].copy()
    X_va = train.loc[va_idx, FEATURES].copy()
    y_va_hard = train.loc[va_idx, TARGET].copy()
    X_test = test[FEATURES].copy()

    y_tr_soft = ALPHA * y_tr_hard + (1 - ALPHA) * teacher_preds[tr_idx]
    y_va_soft = ALPHA * y_va_hard + (1 - ALPHA) * teacher_preds[va_idx]

    for col in ALL_CATS:
      nc = f'{col}_te'
      TE = TargetEncoder(n_folds=5, smooth=10, split_method='random')
      X_tr[nc] = TE.fit_transform(X_tr[col], y_tr_hard)
      X_va[nc] = TE.transform(X_va[col])
      X_test[nc] = TE.transform(X_test[col])

    fold_ensemble_preds = np.zeros(len(va_idx), dtype=np.float32)
    fold_test_ensemble_preds = np.zeros(len(test), dtype=np.float32)

    for model_idx, params in enumerate(top3_params):
        model = RealMLP_TD_Regressor(**params)
        model.fit(X_tr, y_tr_soft, X_val=X_va, y_val=y_va_soft, cat_col_names=ALL_CATS)
        val_pred = model.predict(X_va)
        fold_ensemble_preds += val_pred / len(top3_params)

        if len(test) > 0:
            t_pred = model.predict(X_test)
            fold_test_ensemble_preds += t_pred / len(top3_params)

        score = roc_auc_score(y_va_hard, val_pred)
        print(f"{fold+1:<5} | #{model_idx+1:<4} | {score:.5f}")

        del model, val_pred
        torch.cuda.empty_cache()
        gc.collect()

    oof_preds[va_idx] = fold_ensemble_preds
    test_preds += fold_test_ensemble_preds / 5

    ensemble_score = roc_auc_score(y_va_hard, fold_ensemble_preds)
    print(f"{fold+1:<5} | {'Ens.':<5} | {ensemble_score:.5f}")
    print("-" * 35)

    del X_tr, y_tr_hard, y_tr_soft, X_va, y_va_hard, y_va_soft, fold_ensemble_preds, fold_test_ensemble_preds
    gc.collect()

overall_auc = roc_auc_score(train[TARGET], oof_preds)
print(f"Overall OOF AUC with Distillation: {overall_auc:.6f}")

In [ ]:
pd.DataFrame({'id': train.id, TARGET: oof_preds}).to_csv('oof_realmlp_distil_2.csv', index=False)
pd.DataFrame({'id': test.id, TARGET: test_preds}).to_csv('test_realmlp_distil_2.csv', index=False)